### Installs Dependencies
This cell installs the necessary Python libraries for the project, including `datasets`, `torch`, and `transformers`. These packages are essential for working with the dataset and the CLIP model.

In [12]:
import pickle

with open("../data/val_pairs.pkl", "rb") as f:
    val_pairs = pickle.load(f)

print(len(val_pairs))

3101


## Clip model

### Imports Libraries
This cell imports the required modules and classes, such as `torch`, `CLIPModel`, `CLIPProcessor`, and `load_dataset`. These imports are necessary for loading the model, preprocessing the data, and performing the core logic of the notebook.

In [13]:
import torch
import numpy as np
import faiss
import matplotlib.pyplot as plt

from transformers import CLIPModel, CLIPProcessor
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

### Loads the Dataset
This cell loads the "Good" subset of the "poloclub/diffusiondb" dataset. This dataset contains images and their corresponding textual descriptions, which will be used to test the CLIP model's ability to align images and text.

In [14]:
class FlickrDataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        return self.pairs[idx]

### Defines the Model and Preprocessor
This cell initializes the `CLIPModel` and `CLIPProcessor` from the "openai/clip-vit-base-patch32" pretrained model. The processor is responsible for preparing the data for the model, and the model itself will be used to generate embeddings for the images and text.

In [15]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def collate_fn(batch):
    images, captions = zip(*batch)
    return processor(
        text=list(captions),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )

### Creates the Validation DataLoader
This cell wraps the `FlickrDataset` in a `DataLoader` with a batch size of 32 and no shuffling, using the custom `collate_fn` to preprocess each batch. This loader will be used during evaluation of the standard CLIP and ALIGN-style experiments.

In [16]:
val_loader = DataLoader(
    FlickrDataset(val_pairs),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

### Loads the Pretrained CLIP Model
This cell detects the available device (CUDA, MPS, or CPU) and loads the pretrained `openai/clip-vit-base-patch32` model onto it. The model is set to evaluation mode (`model.eval()`) to disable dropout and batch normalization during inference.

In [17]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model = model.to(device)
model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

### Defines the `compute_recall` Function
This cell defines the `compute_recall` function, which evaluates image-text retrieval performance. It extracts normalized image and text embeddings from the model, computes a pairwise cosine similarity matrix scaled by a temperature parameter, and calculates **R@1**, **R@5**, and **R@10** — the fraction of queries where the correct match appears in the top 1, 5, and 10 results respectively.

In [27]:
def compute_recall(model, loader, temperature=1.0, mode="clip"):
    
    model.eval()
    
    all_image_embeds = []
    all_text_embeds = []
    
    with torch.no_grad():
        for batch in loader:
            
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            
            image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
            text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
            
            all_image_embeds.append(image_embeds)
            all_text_embeds.append(text_embeds)
    
    all_image_embeds = torch.cat(all_image_embeds)
    all_text_embeds = torch.cat(all_text_embeds)
    
    # similarity matrix
    similarity = (all_text_embeds @ all_image_embeds.T) / temperature
    
    similarity_np = similarity.cpu().numpy()
    
    r1 = r5 = r10 = 0
    
    for i in range(len(similarity_np)):
        
        ranked = np.argsort(-similarity_np[i])
        
        if i in ranked[:1]:
            r1 += 1
        
        if i in ranked[:5]:
            r5 += 1
        
        if i in ranked[:10]:
            r10 += 1
    
    N = len(similarity_np)
    
    return r1/N, r5/N, r10/N

### Evaluates CLIP Baseline at Multiple Temperatures
This cell runs the standard CLIP model on the validation set across a range of temperature values (`[1.0, 0.5, 0.1, 0.07, 0.03]`). For each temperature, it prints the **R@1**, **R@5**, and **R@10** scores to observe how the softmax temperature affects retrieval performance.

In [28]:
print("===== CLIP BASELINE =====")

temps = [1.0, 0.5, 0.1, 0.07, 0.03]

for t in temps:
    
    r1, r5, r10 = compute_recall(model, val_loader, temperature=t)
    
    print(f"Temp {t}")
    print(f"R@1: {r1:.4f}  R@5: {r5:.4f}  R@10: {r10:.4f}")
    print()

===== CLIP BASELINE =====
Temp 1.0
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.5
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.1
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.07
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.03
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110



### Evaluates ALIGN-Style Retrieval
This cell defines and runs `compute_recall_align`, an ALIGN-inspired evaluation function. Similar to the CLIP baseline, it computes normalized embeddings and a scaled similarity matrix, but follows the ALIGN methodology of evaluating text-to-image retrieval. Results are printed for each temperature value.

In [29]:
print("===== ALIGN STYLE =====")

def compute_recall_align(model, loader, temperature=1.0):
    
    model.eval()
    
    all_image_embeds = []
    all_text_embeds = []
    
    with torch.no_grad():
        for batch in loader:
            
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            
            image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
            text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
            
            all_image_embeds.append(image_embeds)
            all_text_embeds.append(text_embeds)
    
    all_image_embeds = torch.cat(all_image_embeds)
    all_text_embeds = torch.cat(all_text_embeds)
    
    similarity = (all_text_embeds @ all_image_embeds.T) / temperature
    
    similarity_np = similarity.cpu().numpy()
    
    r1 = r5 = r10 = 0
    
    for i in range(len(similarity_np)):
        
        ranked = np.argsort(-similarity_np[i])
        
        if i in ranked[:1]:
            r1 += 1
        
        if i in ranked[:5]:
            r5 += 1
        
        if i in ranked[:10]:
            r10 += 1
    
    N = len(similarity_np)
    
    return r1/N, r5/N, r10/N


for t in temps:
    
    r1, r5, r10 = compute_recall_align(model, val_loader, temperature=t)
    
    print(f"Temp {t}")
    print(f"R@1: {r1:.4f}  R@5: {r5:.4f}  R@10: {r10:.4f}")
    print()

===== ALIGN STYLE =====
Temp 1.0
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.5
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.1
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.07
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110

Temp 0.03
R@1: 0.4589  R@5: 0.7153  R@10: 0.8110



### Defines DeCLIP-Style Image Augmentations
This cell defines a `transforms.Compose` pipeline called `declips_aug` that applies random resized cropping, horizontal flipping, and color jitter to images. These augmentations are inspired by the DeCLIP approach, which leverages self-supervised signals through augmented views of the same image.

In [30]:
from torchvision import transforms

declips_aug = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(),
])

### Defines the DeCLIP Collate Function
This cell defines `collate_fn_declip`, a custom collate function that applies the DeCLIP augmentation pipeline to each image in the batch before passing it to the processor. This simulates the DeCLIP training strategy of using augmented image views during evaluation.

In [31]:
def collate_fn_declip(batch):
    
    images, captions = zip(*batch)
    
    augmented_images = [declips_aug(img) for img in images]
    
    return processor(
        text=list(captions),
        images=augmented_images,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

### Evaluates DeCLIP-Style Retrieval
This cell runs the pretrained CLIP model on the DeCLIP-augmented validation set across all temperature values. The resulting **R@1**, **R@5**, and **R@10** scores are printed for each temperature, allowing a direct comparison between the standard CLIP, ALIGN-style, and DeCLIP-style evaluation strategies.

In [32]:
val_loader_declip = DataLoader(
    FlickrDataset(val_pairs),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_declip
)

In [33]:
print("===== DECLIP STYLE =====")

for t in temps:
    
    r1, r5, r10 = compute_recall(model, val_loader_declip, temperature=t)
    
    print(f"Temp {t}")
    print(f"R@1: {r1:.4f}  R@5: {r5:.4f}  R@10: {r10:.4f}")
    print()

===== DECLIP STYLE =====
Temp 1.0
R@1: 0.3360  R@5: 0.5946  R@10: 0.6872

Temp 0.5
R@1: 0.3457  R@5: 0.5885  R@10: 0.6898

Temp 0.1
R@1: 0.3386  R@5: 0.5759  R@10: 0.6782

Temp 0.07
R@1: 0.3389  R@5: 0.5779  R@10: 0.6730

Temp 0.03
R@1: 0.3431  R@5: 0.5676  R@10: 0.6604



In [ ]:
# r1, r5, r10 = compute_recall(model, val_loader)

# print("CLIP Baseline")
# print("R@1:", r1)
# print("R@5:", r5)
# print("R@10:", r10)

CLIP Baseline
R@1: 0.45888423089326025
R@5: 0.7152531441470493
R@10: 0.8110287004192196


In [ ]:
# temps = [0.01, 0.03, 0.07, 0.1, 0.5, 1.0]

# results = []

# for t in temps:
#     r1, r5, r10 = compute_recall(model, val_loader, temperature=t)
#     results.append((t, r1, r5, r10))
#     print(f"Temp {t} → R@1: {r1:.4f}, R@5: {r5:.4f}, R@10: {r10:.4f}")

Temp 0.01 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
Temp 0.03 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
Temp 0.07 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
Temp 0.1 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
Temp 0.5 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
Temp 1.0 → R@1: 0.4589, R@5: 0.7153, R@10: 0.8110
